# CubicasA5K — YOLOv8 Floor Plan Detection
**Classes:** door · wall · window  
**Dataset:** 4178 train / 400 val / 400 test  

### Setup instructions before running:
1. Upload your dataset zip to Google Drive (see Cell 3 for folder structure)
2. Runtime → Change runtime type → **T4 GPU**
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
import torch, subprocess

# P100 is sm_60 — PyTorch 2.2+ dropped sm_60 support, so reinstall 2.1 if needed
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    if cap[0] < 7:   # sm_60 (P100) — need older PyTorch
        print(f'GPU capability sm_{cap[0]}{cap[1]} detected — installing PyTorch 2.1 (P100 compatible)...')
        subprocess.run([
            'pip', 'install', '-q',
            'torch==2.1.2+cu118', 'torchvision==0.16.2+cu118',
            '--index-url', 'https://download.pytorch.org/whl/cu118'
        ], check=True)
        print('PyTorch 2.1 installed — please restart session now (Run → Restart & Run All)')
    else:
        print(f'GPU capability sm_{cap[0]}{cap[1]} — PyTorch is compatible, no changes needed')

%pip install -q ultralytics
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')
import torch
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Cell 3: Detect environment (Colab vs Kaggle) and mount Drive if needed ────
import os

# Kaggle sets this env var in all kernel sessions — more reliable than path check
IS_KAGGLE = (
    'KAGGLE_KERNEL_RUN_TYPE' in os.environ or
    os.path.exists('/kaggle/input') or
    os.path.exists('/kaggle/working')
)
IS_COLAB = not IS_KAGGLE

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Google Colab"}')
print(f'KAGGLE_KERNEL_RUN_TYPE: {os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "not set")}')

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Google Drive mounted at /content/drive')
    except Exception as e:
        print(f'Drive mount skipped: {e}')
else:
    print('Kaggle: data comes from /kaggle/input — no Drive mount needed')

In [ ]:
# ── Cell 4: Locate and unzip dataset ─────────────────────────────────────────
import os, glob

EXTRACT_TO = '/kaggle/working/datasets' if IS_KAGGLE else '/content/datasets'
os.makedirs(EXTRACT_TO, exist_ok=True)

if IS_KAGGLE:
    # Find the zip anywhere under /kaggle/input/
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    ZIP_PATH = zips[0] if zips else None
    print(f'Kaggle input files: {os.listdir("/kaggle/input")}')
else:
    ZIP_PATH = '/content/drive/MyDrive/cubicasa5k.zip'

if ZIP_PATH and os.path.exists(ZIP_PATH):
    print(f'Unzipping {ZIP_PATH} ...')
    os.system(f'unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_TO}"')
    print('Done.')
else:
    print(f'No zip found. If already extracted, ignore this.')

print(f'\nContents of {EXTRACT_TO}:')
print(os.listdir(EXTRACT_TO))

In [ ]:
# ── Cell 5: Rewrite data.yaml with absolute paths ────────────────────────────
import yaml, os

DATASET_ROOT  = os.path.join(EXTRACT_TO, 'cubicasa5k-2-6')
colab_yaml_path = '/kaggle/working/data.yaml' if IS_KAGGLE else '/content/data.yaml'

data = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 3,
    'names': ['door', 'wall', 'window']
}

with open(colab_yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f'data.yaml written to {colab_yaml_path}:')
os.system(f'cat {colab_yaml_path}')

train_count = len(os.listdir(os.path.join(DATASET_ROOT, 'train', 'images')))
val_count   = len(os.listdir(os.path.join(DATASET_ROOT, 'valid', 'images')))
print(f'\nTrain images: {train_count}')
print(f'Val images:   {val_count}')

In [ ]:
# ── Cell 6: Auto-select batch size based on GPU VRAM ─────────────────────────
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {vram_gb:.1f} GB')

if vram_gb >= 38:    # A100
    BATCH = 32
    MODEL = 'yolov8m.pt'
    IMGSZ = 1024
elif vram_gb >= 15:  # T4 / V100
    BATCH = 16
    MODEL = 'yolov8m.pt'   # upgraded from yolov8s for better wall recall
    IMGSZ = 1024
else:                # smaller GPU
    BATCH = 8
    MODEL = 'yolov8s.pt'
    IMGSZ = 640

print(f'Selected model : {MODEL}')
print(f'Selected batch : {BATCH}')
print(f'Selected imgsz : {IMGSZ}')

In [ ]:
# ── Cell 7: TRAIN ─────────────────────────────────────────────────────────────
from ultralytics import YOLO

RUN_DIR = '/kaggle/working/runs' if IS_KAGGLE else '/content/runs'

model = YOLO(MODEL)

results = model.train(
    data        = colab_yaml_path,
    epochs      = 80,
    imgsz       = IMGSZ,
    batch       = BATCH,
    device      = 0,
    workers     = 4,
    cache       = True,
    patience    = 20,
    multi_scale = True,
    cos_lr      = True,
    close_mosaic= 15,
    optimizer   = 'AdamW',
    lr0         = 0.001,
    lrf         = 0.01,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
    weight_decay     = 0.0005,
    box         = 7.5,
    cls         = 0.5,
    dfl         = 1.5,
    iou         = 0.7,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.15,
    degrees     = 10.0,
    translate   = 0.1,
    scale       = 0.6,
    shear       = 2.0,
    fliplr      = 0.5,
    flipud      = 0.25,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    perspective = 0.0001,
    project     = RUN_DIR,
    name        = 'cubicasa_yolo',
    exist_ok    = True,
)

In [ ]:
# ── Cell 8: Validate best model ───────────────────────────────────────────────
from ultralytics import YOLO

best_model = YOLO(f'{RUN_DIR}/cubicasa_yolo/weights/best.pt')

metrics = best_model.val(
    data    = colab_yaml_path,
    imgsz   = IMGSZ,
    split   = 'val',
    iou     = 0.45,
    conf    = 0.001,
    device  = 0,
)

print('\n========== RESULTS ==========')
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print('==============================')
for i, name in enumerate(['door', 'wall', 'window']):
    p  = metrics.box.p[i]
    r  = metrics.box.r[i]
    m  = metrics.box.ap50[i]
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f"  {name:6s}  P={p:.3f}  R={r:.3f}  mAP50={m:.3f}  F1={f1:.3f}")

In [ ]:
# ── Cell 9: Show training curves ──────────────────────────────────────────────
from IPython.display import Image, display
import os

run_dir = f'{RUN_DIR}/cubicasa_yolo'

for img in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    path = os.path.join(run_dir, img)
    if os.path.exists(path):
        print(f'--- {img} ---')
        display(Image(path))

In [ ]:
# ── Cell 10: Save best model ──────────────────────────────────────────────────
import shutil, os

if IS_KAGGLE:
    # On Kaggle, /kaggle/working/ is automatically saved as output
    SAVE_DIR = '/kaggle/working/trained_models'
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copy(f'{RUN_DIR}/cubicasa_yolo/weights/best.pt', f'{SAVE_DIR}/best.pt')
    shutil.copy(f'{RUN_DIR}/cubicasa_yolo/weights/last.pt', f'{SAVE_DIR}/last.pt')
    print(f'Saved to {SAVE_DIR} (download from Kaggle Output tab)')
else:
    SAVE_DIR = '/content/drive/MyDrive/cubicasa5k/trained_models'
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copy(f'{RUN_DIR}/cubicasa_yolo/weights/best.pt', f'{SAVE_DIR}/best.pt')
    shutil.copy(f'{RUN_DIR}/cubicasa_yolo/weights/last.pt', f'{SAVE_DIR}/last.pt')
    shutil.copytree(f'{RUN_DIR}/cubicasa_yolo', f'{SAVE_DIR}/run_results', dirs_exist_ok=True)
    print(f'Saved to Google Drive: {SAVE_DIR}')

print(os.listdir(SAVE_DIR))